In [1]:
%run limpieza_data.ipynb

Todos los archivos han sido cargados
dataframe expextativas limpio
dataframe tasa_politica limpio
dataframe indice_precios limpio
dataframe tasa_ibr limpio
dataframe tasa_mercado limpio
Filtro temporal aplicado
Valores del mes seleccioandos
DATAFRAME CONSOLIDADO
Columna fecha eliminada de df_modelo_sin_fecha


In [2]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

## Normalización de datos

In [3]:
n = len(df_modelo_sin_fecha)
train_obs = int(n * 0.70)
val_obs = int(n * 0.85)
test_obs = n - train_obs - val_obs

scaler = StandardScaler()
scaler.fit(df_modelo_sin_fecha[:train_obs])
df_modelo_sin_fecha = scaler.transform(df_modelo_sin_fecha)

ipc_mean = scaler.mean_[2]
ipc_std = scaler.scale_[2]

In [13]:
pasos_dato = 1
contexto = 12
retraso = pasos_dato * (contexto + 1 - 1)
batch_size = 32

class BanrepDataset(Dataset):
    def __init__(self, data, objetivo, contexto, pasos_dato, inicio, final, shuffle = False):
        self.data = data
        self.targets = objetivo
        self.sequence_length = contexto
        self.sampling_rate = pasos_dato
        self.indices = np.arange(inicio, final)
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, indice):
        inicio = self.indices[indice]
        pasos = np.arange(inicio, inicio + self.sequence_length * self.sampling_rate, self.sampling_rate)
        x = self.data[pasos]
        y = self.data[inicio + retraso, self.targets]
        return torch.tensor(x, dtype = torch.float32), torch.tensor(y, dtype = torch.float32)

In [14]:
train_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    objetivo = 2, contexto = contexto, 
    pasos_dato = pasos_dato, 
    inicio = 0, final = train_obs
)

val_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    objetivo = 2, contexto = contexto,
    pasos_dato = pasos_dato,
    inicio = train_obs - retraso,
    final = val_obs - retraso, 
)

test_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    objetivo = 2, contexto = contexto,
    pasos_dato = pasos_dato, 
    inicio = val_obs - retraso,
    final = n - retraso
)

In [15]:
train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_dataloader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

# Validación de dimensiones
for entrada, objetivo in train_dataloader:
    print(f'Dimensiones de entrada: {entrada.shape}')
    print(f'Dimensiones de variable objetivo: {objetivo.shape}')
    break

Dimensiones de entrada: torch.Size([32, 12, 5])
Dimensiones de variable objetivo: torch.Size([32])


In [16]:
def correr_etapa(modelo, lote, criterio, optimizador = None):
    entrenamiento = optimizador is not None
    modelo.train() if entrenamiento else modelo.eval()
    perdida_total = 0.0
    mae_total = 0.0
    n = 0

    with torch.set_grad_enabled(entrenamiento):
        for entrada, objetivo in lote:
            entrada, objetivo = entrada.to(dispositivo), objetivo.to(dispositivo)
            predicciones = modelo(entrada)
            perdida = criterio(predicciones, objetivo)

            if entrenamiento:
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()
            
            perdida_total += perdida
            mae_total = torch.sum(torch.abs(predicciones - objetivo)).item()
            n += entrada.size(0)

    return perdida_total / n, mae_total / n 

def obtener_predicciones(modelo, dataset):
    modelo.eval()
    predicciones_completa = []
    objetivo_completa = []
    lote = Dataloader(dataset, batch_size = 32, shuffle = False)

    with torch.no_grad():
        for entrada, objetivo in lote:
            predicciones = modelo(entrada.to(dispositivo).cpu().numpy())
            predicciones_completa.append(predicciones * ipc_std + ipc_mean)
            objetivo_completa.append(objetivo.numpy())
        
        return np.concatenate(predicciones_completa), np.concatenate(objetivo_completa) 

In [17]:
class mejor_modelo:
    def __init__(self, directorio, vigilante = 'val_mae', comparacion = 'min', mostrar = True):
        self.filepath = directorio
        self.monitor = vigilante
        self.mode = comparacion
        self.verbose = mostrar
        self.best = float('inf') if self.mode == 'min' else float('-inf')

    def paso(self, metricas, modelo):
        valor = metricas[self.monitor]
        mejora = (valor < self.best if self.mode == 'min' else valor > self.best)

        if mejora:
            self.best = valor
            torch.save(modelo.state_dict(), self.filepath)
            if self.verbose:
                print('Mejor modelo guardado')


In [18]:
class parada_lr:
    def __init__ (self, vigilante = 'val_mae', paciencia = 5, cambio_min = 1e-4, comparacion = 'min'):
        self.monitor = vigilante
        self.patience = paciencia
        self.min_delta = cambio_min
        self.mode = comparacion
        self.best = float('-inf') if self.mode == 'min' else float('inf')
        self.counter = 0
        self.debe_parar = False

    def paso(self, metricas, modelo):
        valor = metricas[self.monitor]
        mejora = (valor < self.best if self.mode == 'min' else value > self.best)
        if mejora:
            self.best = valor
            self.counter = 0
        else:
            self.counter +=1
            if self.counter >= self.patience:
                self.debe_parar = True
                print(f'Se detiene Loop debido a no mejora en {self.patience} etapas')


In [19]:
class lr_adaptativo: 
    def __init__(self, optimizador, vigilante = 'val_mae', paciencia = 3, factor = 0.5, mostrar = True):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizador, patience = paciencia, factor = factor, verbose = mostrar
        )
        self.monitor = vigilante

    def paso(self, metricas, modelo = None):
        self.scheduler.step(metricas[self.monitor])


In [20]:
class modelo_gru(nn.Module):
    def __init__(self, caracteristicas, neuronas = 2):
        super().__init__()
        self.gru = nn.GRU(input_size = caracteristicas, hidden_size = neuronas, batch_first = True)
        self.linear = nn.Linear(neuronas, 1)
    
    def forward(self, x):
        resultado, _ = self.gru(x)
        return self.linear(resultado[:, -1, :]).squeeze(-1)

In [21]:
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = modelo_gru(caracteristicas = df_modelo_sin_fecha.shape[-1]).to(dispositivo)
optimizador = torch.optim.Adam(modelo.parameters())
criterio = nn.MSELoss()
muestra = SummaryWriter(log_dir = 'runs/modelo_GRU')

llamadas = [
    mejor_modelo('Mejor_modelo_gru.pt')
]

etapas = 10
for etapa in range(1, etapas + 1):
    perd_entreno, mae_entreno = correr_etapa(modelo, train_dataloader, criterio, optimizador)
    perd_val, val_mae = correr_etapa(modelo, val_dataloader, criterio)

    metricas = {'perd_entreno': perd_entreno, 'mae_entreno': mae_entreno,
                'per_val': perd_val, 'val_mae': val_mae}

    muestra.add_scalars('Perdida', {'entrenamiento': perd_entreno, 'validacion': perd_val}, etapa)
    muestra.add_scalars('MAE', {'entrenamiento': mae_entreno, 'validacion': val_mae}, etapa)

    print(f'Etapa: {etapa:02d} | '  
          f'Perdida entrenamiento: {perd_entreno:.4f}, MAE entrenamiento: {mae_entreno:.4f} | '
          f'Perdida validacion: {perd_val:.4f}, MAE validacion {val_mae:.4f}')
    
    for llamada in llamadas:
        llamada.paso(metricas, modelo) if isinstance(llamada, mejor_modelo) else llamada.paso(metricas)

muestra.close() 

Etapa: 01 | Perdida entrenamiento: 0.0601, MAE entrenamiento: 0.1932 | Perdida validacion: 1.0210, MAE validacion 0.1400
Mejor modelo guardado
Etapa: 02 | Perdida entrenamiento: 0.0577, MAE entrenamiento: 0.1245 | Perdida validacion: 1.0160, MAE validacion 0.1399
Mejor modelo guardado
Etapa: 03 | Perdida entrenamiento: 0.0575, MAE entrenamiento: 0.1801 | Perdida validacion: 1.0111, MAE validacion 0.1399
Mejor modelo guardado
Etapa: 04 | Perdida entrenamiento: 0.0555, MAE entrenamiento: 0.1324 | Perdida validacion: 1.0062, MAE validacion 0.1399
Mejor modelo guardado
Etapa: 05 | Perdida entrenamiento: 0.0553, MAE entrenamiento: 0.2025 | Perdida validacion: 1.0016, MAE validacion 0.1399
Mejor modelo guardado
Etapa: 06 | Perdida entrenamiento: 0.0535, MAE entrenamiento: 0.1442 | Perdida validacion: 0.9968, MAE validacion 0.1399
Mejor modelo guardado
Etapa: 07 | Perdida entrenamiento: 0.0529, MAE entrenamiento: 0.1845 | Perdida validacion: 0.9922, MAE validacion 0.1399
Mejor modelo guardado

In [22]:
modelo.load_state_dict(torch.load('Mejor_modelo_gru.pt', map_location = dispositivo))
_, test_mae = correr_etapa(modelo, test_dataloader, criterio)
print(f'MAE en test: {test_mae:.4f}')

MAE en test: 0.1929


In [25]:
class modelo_gru_multicapa(nn.Module):
    def __init__(self, caracteristicas, neuronas = 2):
        super().__init__()
        self.gru = nn.GRU(input_size = caracteristicas, hidden_size = neuronas, num_layers = 2, batch_first = True)
        self.linear = nn.Linear(neuronas, 1)
    
    def forward(self, x):
        resultado, _ = self.gru(x)
        return self.linear(resultado[:, -1, :]).squeeze(-1)


In [26]:
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = modelo_gru_multicapa(caracteristicas = df_modelo_sin_fecha.shape[-1]).to(dispositivo)
optimizador = torch.optim.Adam(modelo.parameters())
criterio = nn.MSELoss()
muestra = SummaryWriter(log_dir = 'runs/modelo_GRU_multicapa')

llamadas = [
    mejor_modelo('Mejor_modelo_gru_multicapa.pt')
]

etapas = 10
for etapa in range(1, etapas + 1):
    perd_entreno, mae_entreno = correr_etapa(modelo, train_dataloader, criterio, optimizador)
    perd_val, val_mae = correr_etapa(modelo, val_dataloader, criterio)

    metricas = {'perd_entreno': perd_entreno, 'mae_entreno': mae_entreno,
                'per_val': perd_val, 'val_mae': val_mae}

    muestra.add_scalars('Perdida', {'entrenamiento': perd_entreno, 'validacion': perd_val}, etapa)
    muestra.add_scalars('MAE', {'entrenamiento': mae_entreno, 'validacion': val_mae}, etapa)

    print(f'Etapa: {etapa:02d} | '  
          f'Perdida entrenamiento: {perd_entreno:.4f}, MAE entrenamiento: {mae_entreno:.4f} | '
          f'Perdida validacion: {perd_val:.4f}, MAE validacion {val_mae:.4f}')
    
    for llamada in llamadas:
        llamada.paso(metricas, modelo) if isinstance(llamada, mejor_modelo) else llamada.paso(metricas)

muestra.close() 

Etapa: 01 | Perdida entrenamiento: 0.0727, MAE entrenamiento: 0.1822 | Perdida validacion: 1.1487, MAE validacion 0.1523
Mejor modelo guardado
Etapa: 02 | Perdida entrenamiento: 0.0722, MAE entrenamiento: 0.2315 | Perdida validacion: 1.1360, MAE validacion 0.1516
Mejor modelo guardado
Etapa: 03 | Perdida entrenamiento: 0.0695, MAE entrenamiento: 0.1757 | Perdida validacion: 1.1232, MAE validacion 0.1510
Mejor modelo guardado
Etapa: 04 | Perdida entrenamiento: 0.0676, MAE entrenamiento: 0.1734 | Perdida validacion: 1.1107, MAE validacion 0.1503
Mejor modelo guardado
Etapa: 05 | Perdida entrenamiento: 0.0662, MAE entrenamiento: 0.1803 | Perdida validacion: 1.0985, MAE validacion 0.1496
Mejor modelo guardado
Etapa: 06 | Perdida entrenamiento: 0.0647, MAE entrenamiento: 0.1812 | Perdida validacion: 1.0864, MAE validacion 0.1489
Mejor modelo guardado
Etapa: 07 | Perdida entrenamiento: 0.0631, MAE entrenamiento: 0.1820 | Perdida validacion: 1.0748, MAE validacion 0.1483
Mejor modelo guardado

In [27]:
modelo.load_state_dict(torch.load('Mejor_modelo_gru_multicapa.pt', map_location = dispositivo))
_, test_mae = correr_etapa(modelo, test_dataloader, criterio)
print(f'MAE en test: {test_mae:.4f}')

MAE en test: 0.2041
